# YaadDost: Gemma server on Colab
Runs an open-weight **Gemma** model with Ollama on Colab's free GPU and exposes it through a temporary public URL,
so the YaadDost app (on Render, or on your laptop) can use it. No local disk space needed.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.

Heads-up: the URL printed at the end is public to anyone who has it. It is random and dies with the session, but don't share it.

In [ ]:
%%bash
apt-get install -y -qq zstd > /dev/null
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, urllib.request

subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(1)
print("ollama is up")

In [ ]:
# gemma4:e4b is the default. Use gemma4:e2b for a smaller/faster model (then set GEMMA_MODEL=gemma4:e2b in the app).
MODEL = "gemma4:e4b"
!ollama pull {MODEL}

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

In [ ]:
# Start the tunnel. Free Cloudflare quick tunnels sometimes drop, so this cell also defines the checks
# that the keep-alive cell at the bottom uses to notice a dead tunnel and start a new one.
import re, subprocess, time, urllib.error, urllib.request

TUNNEL_LOG = "/content/tunnel.log"
OLLAMA_LOG = "/content/ollama.log"
CLOUDFLARED = "/content/cloudflared"
METRICS = "127.0.0.1:20241"        # cloudflared answers /ready here once it is connected
tunnel = {"proc": None, "url": None}


def _status(url, timeout=3):
    """HTTP status of a GET request, or None if nothing answers."""
    try:
        return urllib.request.urlopen(url, timeout=timeout).status
    except urllib.error.HTTPError as e:
        return e.code
    except Exception:
        return None


def _read(path):
    try:
        return open(path, encoding="utf-8", errors="ignore").read()
    except OSError:
        return ""


def ollama_ok():
    return _status("http://localhost:11434/api/tags") == 200


def tunnel_ok():
    proc = tunnel["proc"]
    if proc is None or proc.poll() is not None:
        return False
    status = _status(f"http://{METRICS}/ready")
    if status is not None:
        return status == 200
    return "Registered tunnel connection" in _read(TUNNEL_LOG)   # metrics port silent: fall back to the log


def start_ollama():
    subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "a"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_ok():
            return True
        time.sleep(1)
    return False


def start_tunnel():
    """Start a fresh tunnel. Returns its public URL, or None if it never came up."""
    subprocess.run(["pkill", "-f", "cloudflared"], check=False)   # remove any older tunnel, even from an earlier cell
    time.sleep(1)
    open(TUNNEL_LOG, "w").close()
    tunnel["url"] = None
    tunnel["proc"] = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434",
         "--metrics", METRICS],
        stdout=open(TUNNEL_LOG, "a"), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", _read(TUNNEL_LOG))
        if m and tunnel_ok():
            tunnel["url"] = m.group(0)
            return tunnel["url"]
        time.sleep(1)
    return None


if globals().get("KEEPALIVE_AUTORUN", True):
    url = start_tunnel()
    print("OLLAMA_URL =", url)


In [ ]:
# 1) Does Gemma itself work? This talks to Ollama on this machine, so no tunnel is involved.
import json, socket, time, urllib.request

def ask(base, timeout=300):
    req = urllib.request.Request(
        base + "/api/chat",
        data=json.dumps({"model": MODEL, "stream": False, "messages": [{"role": "user", "content": "Say hi in Hinglish, one line."}]}).encode(),
        headers={"Content-Type": "application/json"},
    )
    return json.load(urllib.request.urlopen(req, timeout=timeout))["message"]["content"]

print("Gemma (local):", ask("http://localhost:11434"))

# 2) Is the public URL reachable? A brand-new tunnel name can take a minute to reach DNS, and Colab's own
#    resolver is often the last to learn it. Failing here does NOT mean the tunnel is broken for other computers.
host = url.replace("https://", "")
for _ in range(12):
    try:
        socket.getaddrinfo(host, 443)
        print("Gemma (via the public URL):", ask(url))
        break
    except (socket.gaierror, OSError) as e:
        print("waiting for DNS...", e)
        time.sleep(5)
else:
    print("Colab cannot resolve the tunnel name yet. That is common and usually harmless.")
    print("Check from your own PC instead: open  " + url + "/api/tags  in a browser. A page of JSON means it works.")

print()
print("OLLAMA_URL =", url)


## Connect the app
Set `OLLAMA_URL` to the URL printed above:
- **Render:** Dashboard > your service > Environment > `OLLAMA_URL`, then redeploy.
- **Locally:** `set OLLAMA_URL=https://....trycloudflare.com` (PowerShell: `$env:OLLAMA_URL="..."`), then `uvicorn app.main:app --app-dir backend`.

## Keep it alive (last cell)
Free Cloudflare tunnels sometimes drop. The last cell checks every 20 seconds, restarts Ollama or the tunnel if either
died, and prints a **new URL** whenever it had to recreate the tunnel (a new tunnel always has a new address, so update
`OLLAMA_URL` again). Keep this tab open while you use the app.

**Free GPU time is limited.** That cell stops itself after `MAX_MINUTES` (90 by default) and, with `RELEASE_GPU = True`,
disconnects the session so the GPU is released. Stopping a cell with the square button does not free the GPU. Use
*Runtime > Disconnect and delete runtime* when you are done.


In [ ]:
# KEEP-ALIVE: run this cell LAST.
# Every 20 seconds it checks Ollama and the tunnel, restarts whichever died, and prints a NEW url when
# the tunnel had to be recreated (a new tunnel always has a new address).
# Keep this browser tab open while you use the app.
#
# FREE GPU TIME IS LIMITED. This cell keeps the GPU busy, so it stops by itself after MAX_MINUTES, and with
# RELEASE_GPU = True it then disconnects the session so the GPU is freed for you. Change these two lines to suit.
MAX_MINUTES = 90
RELEASE_GPU = True

import time


def _now():
    return time.strftime("%H:%M:%S")


def banner(new_url):
    print("=" * 72, flush=True)
    print(f"[{_now()}] OLLAMA_URL = {new_url}", flush=True)
    print("Set this as OLLAMA_URL in Render (Environment tab) or in your local run.", flush=True)
    print("=" * 72, flush=True)
    try:
        open("/content/ollama_url.txt", "w").write(new_url)
    except OSError:
        pass


def healthy():
    # A tunnel only counts as working if it is connected AND we know its public address.
    return bool(tunnel["url"]) and tunnel_ok()


def supervise(max_ticks=None, check_every=20, max_minutes=None):
    started = time.monotonic()
    failures = 0
    if not ollama_ok():
        print(f"[{_now()}] starting ollama...", flush=True)
        start_ollama()
    current = tunnel["url"] if healthy() else start_tunnel()
    if current:
        banner(current)
    else:
        failures += 1
        print(f"[{_now()}] the tunnel did not come up, will retry", flush=True)

    ticks = 0
    while max_ticks is None or ticks < max_ticks:
        # after failed attempts wait longer (30s, 60s, ... up to 5 min) so we don't hammer Cloudflare
        time.sleep(check_every if not failures else min(300, 30 * failures))
        ticks += 1
        if max_minutes is not None and time.monotonic() - started >= max_minutes * 60:
            print(f"[{_now()}] reached the {max_minutes}-minute limit, stopping to save your free GPU time.", flush=True)
            return "time_limit"
        if not ollama_ok():
            print(f"[{_now()}] ollama stopped, restarting it...", flush=True)
            start_ollama()
        if healthy():
            failures = 0
            if ticks % 15 == 0:
                print(f"[{_now()}] still running: ollama ok, tunnel ok ({tunnel['url']})", flush=True)
            continue
        print(f"[{_now()}] the tunnel is down, starting a new one...", flush=True)
        new_url = start_tunnel()
        if new_url:
            failures = 0
            banner(new_url)
        else:
            failures += 1
            print(f"[{_now()}] could not start a tunnel (attempt {failures}), will retry", flush=True)


if globals().get("KEEPALIVE_AUTORUN", True):
    if supervise(max_minutes=MAX_MINUTES) == "time_limit" and RELEASE_GPU:
        print("Releasing the GPU now. The tunnel stops with it.", flush=True)
        from google.colab import runtime
        runtime.unassign()
